In [1]:
import os
import pandas as pd
import numpy as np

%run milp_engine.ipynb

In [ ]:
# =====================================================
# 1. Ablation settings
# =====================================================

ABLATIONS = {

    "Full": {

        "turnover": 20,

        "balance": 0.10

    },

    "NoTurnover": {

        "turnover": 0,

        "balance": 0.10

    },

    "NoBalance": {

        "turnover": 20,

        "balance": 0.00

    },

    "NoTurnover_NoBalance": {

        "turnover": 0,

        "balance": 0.00

    }

}


# =====================================================
# 2. Lambda grid
# =====================================================

lambda_list = np.round(

    np.linspace(

        0,

        1,

        11

    ),

    2

)


# =====================================================
# 3. Load RF prediction file
# =====================================================

df = pd.read_csv(

    "Prescriptive/RF_prescriptive_optimizer_inputs.csv"

)


# =====================================================
# 4. Load fixed cohort
# =====================================================

cohort_ids = pd.read_csv(

    "Cohort/fixed_cohort.csv"

)


cohort = df[

    df["LOG_ID"].isin(

        cohort_ids["LOG_ID"]

    )

].copy()


# =====================================================
# 5. Force identical order
# =====================================================

cohort["LOG_ID"] = pd.Categorical(

    cohort["LOG_ID"],

    categories=cohort_ids["LOG_ID"],

    ordered=True

)


cohort = (

    cohort

    .sort_values(

        "LOG_ID"

    )

    .reset_index(

        drop=True

    )

)


# =====================================================
# 6. Safety check
# =====================================================

assert len(cohort) == len(cohort_ids), (

    "Cohort mismatch detected."

)


print(

    f"Fixed cohort loaded: {len(cohort)} cases."

)


# =====================================================
# 7. Run ablation study
# =====================================================

records = []

for exp_name, setting in ABLATIONS.items():

    print(

        "\n"

        + "=" * 60

    )

    print(

        f"Running experiment: {exp_name}"

    )

    print(

        "=" * 60

    )

    for lam in lambda_list:

        print(

            f"Lambda = {lam:.1f}"

        )

        result = solve_or_milp(

            cohort,

            lam,

            turnover=setting["turnover"],

            balance_weight=setting["balance"]

        )

        if result is None:

            print(

                "Solver failed."

            )

            continue

        result["Experiment"] = exp_name

        result["Turnover"] = setting["turnover"]

        result["Balance_Weight"] = setting["balance"]

        records.append(result)


# =====================================================
# 8. Export results
# =====================================================

ablation_df = pd.DataFrame(

    records

)


os.makedirs(

    "Results",

    exist_ok=True

)


output_file = (

    "Results/ablation.csv"

)


ablation_df.to_csv(

    output_file,

    index=False

)


print(

    "\n"

    + "=" * 60

)

print(

    "Ablation study completed."

)

print(

    f"Saved to: {output_file}"

)

print(

    "=" * 60

)

print(

    "\nPreview:\n"

)

print(

    ablation_df.head()

)

Fixed cohort loaded: 12 cases.

Running experiment: Full
Lambda = 0.0
Set parameter Username
Set parameter LicenseID to value 2835369
Academic license - for non-commercial use only - expires 2027-06-15
Lambda = 0.1
Lambda = 0.2
Lambda = 0.3
Lambda = 0.4
Lambda = 0.5
Lambda = 0.6
Lambda = 0.7
Lambda = 0.8
Lambda = 0.9
Lambda = 1.0

Running experiment: NoTurnover
Lambda = 0.0
Lambda = 0.1
Lambda = 0.2
Lambda = 0.3
Lambda = 0.4
Lambda = 0.5
Lambda = 0.6
Lambda = 0.7
Lambda = 0.8
Lambda = 0.9
Lambda = 1.0

Running experiment: NoBalance
Lambda = 0.0
Lambda = 0.1
Lambda = 0.2
Lambda = 0.3
Lambda = 0.4
Lambda = 0.5
Lambda = 0.6
Lambda = 0.7
Lambda = 0.8
Lambda = 0.9
Lambda = 1.0

Running experiment: NoTurnover_NoBalance
Lambda = 0.0
Lambda = 0.1
Lambda = 0.2
Lambda = 0.3
Lambda = 0.4
Lambda = 0.5
Lambda = 0.6
Lambda = 0.7
Lambda = 0.8
Lambda = 0.9
Lambda = 1.0

Ablation study completed.
Saved to: Results/ablation.csv

Preview:

   Lambda    Objective    Overtime  Idle    Makespan  Sched_Util 